# Kaggle Utilities — Project Ouroboros

This notebook replaces the manual `/kaggle/input/...` copy-and-rename flow. It can:

- load Kaggle secrets safely
- hydrate the persistent Hugging Face cache from the attached `ouroboros-cache` dataset
- install the cached Mamba wheels
- pull the latest repo files from GitHub into `/kaggle/working`
- inspect local vs Hugging Face checkpoints
- launch Stage 1 with the latest local-or-HF checkpoint selection logic now built into `pretrain.py`
- launch Stage 2 with auto-detected Dual T4 DDP, timeout-safe checkpoint exit, and Stage-2-first resume logic built into `train_sft.py`


In [ ]:
# AUTO-GENERATED BY DILOCO COORDINATOR. DO NOT EDIT IN REPO.
import base64
import json
import os
import zlib
_DISPATCH_PAYLOAD = "eNq9kk+TmkAQxb+L56yCiuLeAIf/MDjOaMxlChhEXGFQQDem8t0DtdlNzJpULsmx6/1ev5nu/tKbWy7UIF1D5ABErXnvsaf0PvQMk2LoAL8t06zeNREtw5qKohKonxRnI2zMyW7lspWkgpR+PrsjZzc+7wtWmdweqYQvtKl5AEG5FrXCrAP1WF+ObJHlnu74U6L48pDwzBgfuigLm0T9b3Gm/ha121JP9461n+2v+waq6FBid2V9tJZLG6e2Z8Wmdu4sxDAs39AVDdCfn/q3fkgQVCGCS6oQDClGlmEABLpZizfy92VAgttduKjVhf70N0RAcFt1zKkpqgHLDjzm99ilZfiKSxEIYAuz5JxUu3a6A9iceMRPvLprwgoGr551kp0YaooiOf3Z9esV/QC6TlSDnmfhVovi7XAcS7HMJkkoCKIsCVHIhGgaDVkiSkNxto3YLJ68b4GA3vrzMCveawS53VLquqweB4OXO+rHPB/c+XO/lW86rBV/rtIAQRto3RP5K/lQ1WGajB72YR6FreUFVAKLOmDTgpewYBE9i5Sk12vmS/aTDOw8l/M0L6pJaNQRFinzg+CyMUPd4EltLvx5yGdFSZBcNs08uD6XzJzN0uQg+Vi4WE/C5S3oX4fcbOzrN/VSSRA="
_DISPATCH_ENV = json.loads(zlib.decompress(base64.b64decode(_DISPATCH_PAYLOAD)).decode('utf-8'))
for _dispatch_key, _dispatch_value in _DISPATCH_ENV.items():
    if _dispatch_value is None:
        continue
    os.environ[str(_dispatch_key)] = str(_dispatch_value)
del _DISPATCH_ENV, _DISPATCH_PAYLOAD
print('[dispatch] Bound notebook to DiLoCo worker A')


In [1]:
import os
import shutil
from pathlib import Path

!cp -r /kaggle/input/datasets/weirdrunner007/ouroboros-cache/* /kaggle/working/

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

In [2]:
from __future__ import annotations

import os

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None


def _normalize_text(value: object | None, *, uppercase: bool = False) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.upper() if uppercase else text


_secret_client = None


def _get_secret_client():
    global _secret_client
    if _secret_client is False:
        return None
    if _secret_client is None:
        if UserSecretsClient is None:
            _secret_client = False
            return None
        try:
            _secret_client = UserSecretsClient()
        except Exception:
            _secret_client = False
            return None
    return _secret_client if _secret_client is not False else None


def _optional_secret(name: str) -> str | None:
    client = _get_secret_client()
    if client is None:
        return None
    try:
        value = client.get_secret(name)
    except Exception:
        return None
    return _normalize_text(value)


def _resolve_value(
    env_names: tuple[str, ...],
    secret_names: tuple[str, ...] = (),
    *,
    uppercase: bool = False,
) -> str | None:
    for env_name in env_names:
        value = _normalize_text(os.environ.get(env_name), uppercase=uppercase)
        if value is not None:
            return value
    for secret_name in secret_names:
        value = _normalize_text(_optional_secret(secret_name), uppercase=uppercase)
        if value is not None:
            return value
    return None


HF_TOKEN = _resolve_value(("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"), ("HF_TOKEN",))
WANDB_KEY = _resolve_value(("WANDB_API_KEY", "WANDB_KEY"), ("WANDB_KEY",))
GITHUB_TOKEN = _resolve_value(("GITHUB_TOKEN", "GH_TOKEN"), ("GITHUB_TOKEN", "GH_TOKEN"))
DILOCO_WORKER_ID = _resolve_value(
    ("DILOCO_WORKER_ID", "OUROBOROS_DILOCO_WORKER_ID", "WORKER_ID"),
    ("DILOCO_WORKER_ID",),
    uppercase=True,
)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    os.environ["WANDB_KEY"] = WANDB_KEY
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    os.environ["GH_TOKEN"] = GITHUB_TOKEN
if DILOCO_WORKER_ID:
    os.environ["DILOCO_WORKER_ID"] = DILOCO_WORKER_ID
    os.environ.setdefault("OUROBOROS_DILOCO_WORKER_ID", DILOCO_WORKER_ID)
    os.environ.setdefault("WORKER_ID", DILOCO_WORKER_ID)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print({
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")),
    "WANDB_KEY": bool(os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_KEY")),
    "GITHUB_TOKEN": bool(os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")),
    "DILOCO_WORKER_ID": bool(
        os.environ.get("DILOCO_WORKER_ID")
        or os.environ.get("OUROBOROS_DILOCO_WORKER_ID")
        or os.environ.get("WORKER_ID")
    ),
})


In [3]:
from __future__ import annotations

import base64
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/deveshpat/Ouroboros.git"
DEFAULT_REPO_REF = "main"

REPO_URL = (os.environ.get("OUROBOROS_REPO_URL") or DEFAULT_REPO_URL).strip()
REPO_REF = (os.environ.get("OUROBOROS_REPO_REF") or DEFAULT_REPO_REF).strip()
REPO_COMMIT = (os.environ.get("OUROBOROS_REPO_COMMIT") or "").strip()
REPO_DIR = Path("/kaggle/working/ouroboros_repo")
TARGET_DIR = Path("/kaggle/working")
FILES_TO_COPY = [
    "jamba_coconut_finetune.py",
]


def run(
    cmd: list[str],
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
        capture_output=True,
    )


def git_env(repo_url: str) -> dict[str, str]:
    env = os.environ.copy()
    token = (os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or "").strip()
    if token and repo_url.startswith("https://") and "github.com" in repo_url:
        basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
        env["GIT_CONFIG_COUNT"] = "1"
        env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
        env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {basic}"
    return env


def ensure_repo(repo_url: str, repo_dir: Path, env: dict[str, str]) -> None:
    if not repo_dir.exists():
        run(["git", "clone", "--filter=blob:none", repo_url, str(repo_dir)], env=env)
    else:
        run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, env=env)
        run(["git", "clean", "-fd"], cwd=repo_dir, env=env)


def fetch_and_checkout(repo_dir: Path, ref: str, commit: str, env: dict[str, str]) -> str:
    ref = (ref or "").strip()
    commit = (commit or "").strip()

    fetched = False
    if ref:
        for fetch_cmd in (
            ["git", "fetch", "--depth", "1", "origin", ref],
            ["git", "fetch", "--depth", "1", "origin", f"refs/heads/{ref}"],
            ["git", "fetch", "--depth", "1", "origin", f"refs/tags/{ref}"],
        ):
            result = run(fetch_cmd, cwd=repo_dir, env=env, check=False)
            if result.returncode == 0:
                fetched = True
                break

    if not fetched:
        run(["git", "fetch", "origin"], cwd=repo_dir, env=env)

    run(["git", "checkout", "--force", "--detach", "FETCH_HEAD"], cwd=repo_dir, env=env)

    if commit:
        commit_result = run(["git", "checkout", "--force", "--detach", commit], cwd=repo_dir, env=env, check=False)
        if commit_result.returncode != 0:
            run(["git", "fetch", "origin"], cwd=repo_dir, env=env)
            commit_result = run(["git", "checkout", "--force", "--detach", commit], cwd=repo_dir, env=env, check=False)
            if commit_result.returncode != 0:
                stderr = (commit_result.stderr or commit_result.stdout or "").strip()
                raise RuntimeError(f"Failed to checkout OUROBOROS_REPO_COMMIT={commit!r}: {stderr}")

    return run(["git", "rev-parse", "HEAD"], cwd=repo_dir, env=env).stdout.strip()


def copy_repo_files(repo_dir: Path, target_dir: Path, file_names: list[str]) -> list[str]:
    copied: list[str] = []
    for name in file_names:
        src = repo_dir / name
        if not src.exists():
            print(f"[warn] missing in repo: {name}")
            continue
        dst = target_dir / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)
    return copied


git_environment = git_env(REPO_URL)
ensure_repo(REPO_URL, REPO_DIR, git_environment)
commit = fetch_and_checkout(REPO_DIR, REPO_REF, REPO_COMMIT, git_environment)
copied = copy_repo_files(REPO_DIR, TARGET_DIR, FILES_TO_COPY)
metadata = {
    "repo_ref": REPO_REF,
    "requested_commit": REPO_COMMIT or None,
    "commit": commit,
    "copied_files": copied,
    "synced_at_utc": datetime.now(timezone.utc).isoformat(),
}
(Path("/kaggle/working/repo_sync_metadata.json")).write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))


In [4]:
# Optional: login to Weights & Biases only when credentials exist.
import os

try:
    import wandb
except Exception as exc:
    print(f"wandb import: skipped ({type(exc).__name__}: {exc})")
else:
    wandb_key = (os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_KEY") or "").strip()
    if wandb_key:
        try:
            os.environ["WANDB_API_KEY"] = wandb_key
            os.environ["WANDB_KEY"] = wandb_key
            wandb.login(key=wandb_key)
            print("wandb login: OK")
        except Exception as exc:
            print(f"wandb login: skipped after failure ({type(exc).__name__}: {exc})")
    else:
        print("wandb login: skipped (W&B credentials not set)")


In [5]:
# ── Cell 5 — DiLoCo Worker Training ─────────────────────────────────────────
#
# Resolution order for the worker id:
#   1. DILOCO_WORKER_ID / OUROBOROS_DILOCO_WORKER_ID env var
#   2. Kaggle secret DILOCO_WORKER_ID (best effort)
#
# This keeps manual runs working while letting the coordinator inject the
# worker id and runtime configuration directly into the staged notebook.

from __future__ import annotations

import os
import shlex
import subprocess
import sys

import torch

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None

VALID_WORKER_IDS = {"A", "B", "C"}


def _normalize_text(value: object | None, *, uppercase: bool = False) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.upper() if uppercase else text


def _optional_worker_secret() -> str | None:
    if UserSecretsClient is None:
        return None
    try:
        return _normalize_text(UserSecretsClient().get_secret("DILOCO_WORKER_ID"), uppercase=True)
    except Exception as exc:
        print(
            "[cell5] Kaggle secret DILOCO_WORKER_ID unavailable "
            f"({type(exc).__name__}: {exc}). Falling back to env-based resolution."
        )
        return None


def _resolve_worker_id() -> str:
    for env_name in ("DILOCO_WORKER_ID", "OUROBOROS_DILOCO_WORKER_ID", "WORKER_ID"):
        worker_id = _normalize_text(os.environ.get(env_name), uppercase=True)
        if worker_id is None:
            continue
        if worker_id not in VALID_WORKER_IDS:
            raise ValueError(
                f"DILOCO_WORKER_ID must be one of {sorted(VALID_WORKER_IDS)} — got {worker_id!r}"
            )
        return worker_id

    worker_id = _optional_worker_secret()
    if worker_id is not None:
        if worker_id not in VALID_WORKER_IDS:
            raise ValueError(
                f"DILOCO_WORKER_ID must be one of {sorted(VALID_WORKER_IDS)} — got {worker_id!r}"
            )
        return worker_id

    raise RuntimeError(
        "Could not resolve a DiLoCo worker id. Set DILOCO_WORKER_ID in the environment "
        "or define a Kaggle secret named DILOCO_WORKER_ID with value A/B/C."
    )


def _env_text(name: str, default: str | None = None) -> str | None:
    value = _normalize_text(os.environ.get(name))
    return value if value is not None else default


def _env_flag(name: str, default: bool) -> bool:
    value = _normalize_text(os.environ.get(name))
    if value is None:
        return default
    return value.lower() not in {"0", "false", "no", "off", "disabled"}


WORKER_ID = _resolve_worker_id()
os.environ["DILOCO_WORKER_ID"] = WORKER_ID
os.environ.setdefault("OUROBOROS_DILOCO_WORKER_ID", WORKER_ID)
os.environ.setdefault("WORKER_ID", WORKER_ID)

hf_token = _env_text("HF_TOKEN") or _env_text("HUGGINGFACE_HUB_TOKEN")
if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is required for DiLoCo mode. Add the Kaggle secret HF_TOKEN or export HF_TOKEN before running cell 5."
    )
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

if not torch.cuda.is_available():
    raise RuntimeError("DiLoCo worker training requires a CUDA GPU, but none were detected.")

gpu_count = torch.cuda.device_count()
nproc = 2 if gpu_count >= 2 else 1

data_dir = _env_text("OUROBOROS_DATA_DIR", "data/coconut_v1")
use_4bit = _env_flag("OUROBOROS_USE_4BIT", True)
stage_0_epochs = _env_text("OUROBOROS_STAGE_0_EPOCHS", "1")
epochs_per_stage = _env_text("OUROBOROS_EPOCHS_PER_STAGE", "1")
max_stage = _env_text("OUROBOROS_MAX_STAGE", "10")
batch_size = _env_text("OUROBOROS_BATCH_SIZE", "4")
grad_accum = _env_text("OUROBOROS_GRAD_ACCUM", "8")
val_batch_size = _env_text("OUROBOROS_VAL_BATCH_SIZE", "2")
val_skip_buffer_minutes = _env_text("OUROBOROS_VAL_SKIP_BUFFER_MINUTES", "60")
session_timeout_hours = _env_text("OUROBOROS_SESSION_TIMEOUT_HOURS", "12.0")
graceful_exit_buffer_minutes = _env_text("OUROBOROS_GRACEFUL_EXIT_BUFFER_MINUTES", "20")
outer_lr = _env_text("OUROBOROS_DILOCO_OUTER_LR", "0.7")
state_repo = _env_text("OUROBOROS_DILOCO_STATE_REPO", "WeirdRunner/Ouroboros")
signal_repo = _env_text("OUROBOROS_DILOCO_SIGNAL_REPO", "deveshpat/Ouroboros")
output_dir = _env_text("OUROBOROS_DILOCO_OUTPUT_DIR", "runs/diloco")
wandb_project = _env_text("OUROBOROS_WANDB_PROJECT", "ouroboros-stage3-jamba")
wandb_entity = _env_text("OUROBOROS_WANDB_ENTITY")
requested_wandb_mode = _env_text("OUROBOROS_WANDB_MODE")
push_to_hub = _env_flag("OUROBOROS_PUSH_TO_HUB", True)

wandb_creds_present = bool(os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_KEY"))
effective_wandb_mode = requested_wandb_mode or "online"
if not wandb_creds_present and effective_wandb_mode == "online":
    effective_wandb_mode = "disabled"
    print("[cell5] WANDB credentials not detected; forcing --wandb_mode disabled.")

cmd = [sys.executable]
if nproc > 1:
    cmd.extend(["-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}"])

cmd.extend([
    "jamba_coconut_finetune.py",
    "--data_dir", data_dir,
    "--stage_0_epochs", stage_0_epochs,
    "--epochs_per_stage", epochs_per_stage,
    "--max_stage", max_stage,
    "--batch_size", batch_size,
    "--grad_accum", grad_accum,
    "--val_batch_size", val_batch_size,
    "--val_skip_buffer_minutes", val_skip_buffer_minutes,
    "--session_timeout_hours", session_timeout_hours,
    "--graceful_exit_buffer_minutes", graceful_exit_buffer_minutes,
    "--diloco_mode",
    "--diloco_worker_id", WORKER_ID,
    "--diloco_outer_lr", outer_lr,
    "--diloco_state_repo", state_repo,
    "--diloco_signal_repo", signal_repo,
    "--output_dir", output_dir,
    "--wandb_project", wandb_project,
    "--wandb_mode", effective_wandb_mode,
])

if use_4bit:
    cmd.append("--use_4bit")
if push_to_hub:
    cmd.append("--push_to_hub")
if wandb_entity:
    cmd.extend(["--wandb_entity", wandb_entity])

print(f"[cell5] Launching as Worker {WORKER_ID} | GPUs={gpu_count} | nproc={nproc}")
cmd_str = " ".join(shlex.quote(part) for part in cmd)
print("[cell5] Command:", cmd_str)
import os as _os
_os.system(cmd_str)